#Bronze Layer Profiling Analysis

######Seeing what needs to be transformed cleaned in silver layer

In [0]:
spark.sql("USE CATALOG ecommerce")
spark.sql("USE SCHEMA bronze")

#####===============================
##CUSTOMERS TABLE PROFILING
#####===============================

####Load Customers Table

In [0]:
customers = spark.table("ecommerce.bronze.customers")

print("Total rows:", customers.count())

display(customers)

In [0]:
customers.printSchema()

In [0]:
from pyspark.sql.functions import col

customers.select([
    col(c).isNull().alias(c)
    for c in customers.columns
]).groupBy().sum().show()

In [0]:
customers.groupBy("customer_id") \
.count() \
.filter("count > 1") \
.show()

In [0]:
customers.groupBy("customer_unique_id") \
.count() \
.filter("count > 1") \
.show()

In [0]:
from pyspark.sql.functions import trim

for c in customers.columns:
    space_count = customers.filter(col(c) != trim(col(c))).count()
    print(c, "space issues:", space_count)

In [0]:
display(customers.select("customer_city").distinct())

In [0]:
display(customers.select("customer_state").distinct())

In [0]:
customers.groupBy("customer_state") \
.count() \
.orderBy("count", ascending=False) \
.show()

In [0]:
customers.select("ingestion_timestamp","source_file").show(10)

#####===============================
##PRODUCTS TABLE PROFILING
#####===============================

In [0]:
products = spark.table("ecommerce.bronze.products")

print("Total rows:", products.count())

display(products)

In [0]:
products.printSchema()

In [0]:
products.select([
    col(c).isNull().alias(c)
    for c in products.columns
]).groupBy().sum().show()

In [0]:
no_negatives_allowed = ["product_weight_g","product_length_cm","product_height_cm","product_photos_qty"]
for n in no_negatives_allowed:
    negative_count = products.filter(col(n) < 0).count()
    print(n, "negative issues:", negative_count)


In [0]:
products.groupBy("product_category_name") \
.count() \
.orderBy("count", ascending=False) \
.show()

In [0]:
display(products.filter("product_category_name IS NULL"))

#####===============================
##ORDER ITEMS TABLE PROFILING
#####===============================

In [0]:
order_items = spark.table("ecommerce.bronze.order_items")

print("Total rows:", order_items.count())

display(order_items)

In [0]:
order_items.printSchema()

In [0]:
order_items.groupBy("order_id","order_item_id") \
.count() \
.filter("count > 1") \
.show()

In [0]:
no_negatives_allowed = ["price","freight_value"]
for n in no_negatives_allowed:
    negative_count = order_items.filter(col(n) < 0).count()
    print(n, "negative issues:", negative_count)

In [0]:
order_items.describe("price","freight_value").show()

#####===============================
##PAYMENTS TABLE PROFILING
#####===============================

In [0]:
payments = spark.table("ecommerce.bronze.payments")

print("Total rows:", payments.count())

display(payments)

In [0]:
payments.printSchema()

In [0]:
payments.select("payment_type") \
.distinct() \
.show()

In [0]:
from pyspark.sql.functions import col
no_negatives_allowed = ["payment_installments","payment_value"]
for n in no_negatives_allowed:
    negative_count = payments.filter(col(n) < 0).count()
    print(n, "negative issues:", negative_count)

In [0]:
payments.groupBy("payment_installments") \
.count() \
.show()

In [0]:
payments.groupBy("order_id").count().orderBy("count", ascending=False).show()

In [0]:
payments.groupBy("order_id","payment_sequential") \
.count() \
.filter("count > 1") \
.show()

#####===============================
##REVIEWS TABLE PROFILING
#####===============================

In [0]:
reviews = spark.table("ecommerce.bronze.reviews")

print("Total rows:", reviews.count())

display(reviews)

In [0]:
reviews.printSchema()

In [0]:
reviews.filter("review_score < 1 OR review_score > 5").show()

In [0]:
from pyspark.sql.functions import col, sum

df_reviews = reviews.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in reviews.columns
])

display(df_reviews)

In [0]:
display(reviews.groupBy("review_id") \
.count() \
.filter("count > 1"))

In [0]:
from pyspark.sql.functions import length

display(reviews.filter(length("review_id") > 40))

In [0]:
display(reviews.filter(col("review_score").cast("int").isNull() & col("review_score").isNotNull()))

In [0]:
from pyspark.sql.functions import trim

for c in reviews.columns:
    
    dirty = reviews.filter(col(c) != trim(col(c))).count()
    
    print(c, "rows with spaces:", dirty)

In [0]:
reviews.filter(length("review_comment_message") > 1000).show()

In [0]:
display(reviews.select(
    "review_creation_date",
    "review_answer_timestamp"
).describe())

In [0]:
orders = spark.table("ecommerce.bronze.orders")

display(reviews.join(
    orders,
    reviews.order_id == orders.order_id,
    "left_anti"
))

In [0]:
display(reviews.filter(
    col("review_id").isNull() &
    col("order_id").isNull() &
    col("review_score").isNull()
))

In [0]:
display(reviews.groupBy("review_score") \
.count() \
.orderBy("review_score"))

In [0]:
display(reviews.filter(
    col("review_comment_message").rlike("[^\\x00-\\x7F]")
))

In [0]:
display(reviews.filter(
    col("review_id").rlike("[a-zA-Z ]{10,}")
))

#####===============================
##REVIEWS_PARSED TABLE PROFILING
#####===============================

In [0]:
reviews_parsed = spark.table("ecommerce.bronze.reviews_parsed")

print("Total rows:", reviews_parsed.count())

display(reviews_parsed)

In [0]:
reviews_parsed.printSchema()

In [0]:
display(reviews_parsed.filter("review_score < 1 OR review_score > 5"))

In [0]:
from pyspark.sql.functions import col, sum

df_reviews_parsed = reviews_parsed.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in reviews_parsed.columns
])

display(df_reviews_parsed)

In [0]:
display(reviews_parsed.groupBy("review_id") \
.count() \
.filter("count > 1"))

In [0]:
from pyspark.sql.functions import length

display(reviews_parsed.filter(length("review_id") > 40))

In [0]:
display(reviews_parsed.filter(col("review_score").cast("int").isNull() & col("review_score").isNotNull()))

In [0]:
from pyspark.sql.functions import trim

for c in reviews_parsed.columns:
    
    dirty = reviews_parsed.filter(col(c) != trim(col(c))).count()
    
    print(c, "rows with spaces:", dirty)

In [0]:
display(reviews_parsed.select(
    "review_creation_date",
    "review_answer_timestamp"
).describe())

In [0]:
orders = spark.table("ecommerce.bronze.orders")

display(reviews_parsed.join(
    orders,
    reviews_parsed.order_id == orders.order_id,
    "left_anti"
))

In [0]:
display(reviews_parsed.filter(
    col("review_id").rlike("[a-zA-Z ]{10,}")
))

#####===============================
##SELLERS TABLE PROFILING
#####===============================

In [0]:
sellers = spark.table("ecommerce.bronze.sellers")

print("Total rows:", sellers.count())

display(sellers)

In [0]:
sellers.groupBy("seller_id") \
.count() \
.filter("count > 1") \
.show()

In [0]:
display(sellers.groupBy("seller_city") \
.count() \
.orderBy("count", ascending=False))

In [0]:
sellers.printSchema()

In [0]:
from pyspark.sql.functions import col

sellers.select([
    col(c).isNull().alias(c)
    for c in sellers.columns
]).groupBy().sum().show()

In [0]:
from pyspark.sql.functions import trim

for c in sellers.columns:
    space_count = sellers.filter(col(c) != trim(col(c))).count()
    print(c, "space issues:", space_count)

In [0]:
display(sellers.select("seller_city").distinct())

In [0]:
display(sellers.select("seller_state").distinct())

In [0]:
sellers.groupBy("seller_state") \
.count() \
.orderBy("count", ascending=False) \
.show()

In [0]:
sellers.select("ingestion_timestamp","source_file").show(10)

#####===============================
##GEOLOCATION TABLE PROFILING
#####===============================

In [0]:
geo = spark.table("ecommerce.bronze.geolocation")

print("Total rows:", geo.count())

display(geo)

In [0]:
geo.filter("geolocation_lat IS NULL").count()

geo.filter("geolocation_lng IS NULL").count()

geo.filter("geolocation_zip_code_prefix IS NULL").count()

In [0]:
display(geo.select("geolocation_city").distinct())

In [0]:
display(geo.select("geolocation_state").distinct())

In [0]:
from pyspark.sql.functions import trim

no_spaces_allowed = ["geolocation_city","geolocation_state"]
for c in no_spaces_allowed:
    space_count = geo.filter(col(c) != trim(col(c))).count()
    print(c, "space issues:", space_count)

In [0]:
display(geo.groupBy("geolocation_city") \
.count() \
.orderBy("count", ascending=False))

In [0]:
display(geo.groupBy("geolocation_state") \
.count() \
.orderBy("count", ascending=False))

In [0]:
display(geo.groupBy(
"geolocation_zip_code_prefix",
"geolocation_lat",
"geolocation_lng"
).count().filter("count > 1"))

#####===============================
##CATEGORY TABLE PROFILING
#####===============================

In [0]:
category = spark.table("ecommerce.bronze.category")

display(category)

In [0]:
category.printSchema()

In [0]:
from pyspark.sql.functions import trim

no_spaces_allowed = ["product_category_name","product_category_name_english"]
for c in no_spaces_allowed:
    space_count = category.filter(col(c) != trim(col(c))).count()
    print(c, "space issues:", space_count)

In [0]:
category.filter("product_category_name IS NULL").count()

category.filter("product_category_name_english IS NULL").count()

In [0]:
category.groupBy("product_category_name") \
.count() \
.filter("count > 1") \
.show()

#####===============================
##ORDERS TABLE PROFILING
#####===============================

In [0]:
orders = spark.table("ecommerce.bronze.orders")

display(orders)

In [0]:
orders.printSchema()

In [0]:
no_null_allowed = ["order_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"]

for c in no_null_allowed:
    null_count = orders.filter(col(c).isNull()).count()
    print(c, "null issues:", null_count)



In [0]:
display(
    orders.groupBy("order_id") \
           .count() \
           .filter("count > 1")
)

In [0]:
customers = spark.table("ecommerce.bronze.customers")

display(orders.join(
    customers,
    orders.customer_id == customers.customer_id,
    "left_anti"
))

In [0]:
orders.select("order_status").distinct().show()

In [0]:
display(orders.filter(
    col("order_approved_at") < col("order_purchase_timestamp")
))

In [0]:
display(orders.filter(
    col("order_delivered_customer_date") < col("order_delivered_carrier_date")
))

In [0]:
display(orders.filter(
    col("order_delivered_carrier_date") < col("order_approved_at")
))

In [0]:
orders.filter(
    col("order_delivered_customer_date") > col("order_estimated_delivery_date")
).count()

In [0]:
from pyspark.sql.functions import current_timestamp

display(orders.filter(
    col("order_purchase_timestamp") > current_timestamp()
))

In [0]:
from pyspark.sql.functions import trim

for c in ["order_id","customer_id","order_status"]:
    
    spaces_count = orders.filter(col(c) != trim(col(c))).count()
    
    print(c,"space_issues",spaces_count)

In [0]:
display(orders.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("order_purchase_timestamp").isNull()
))

In [0]:
from pyspark.sql.functions import datediff

display(orders.select(
    datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    ).alias("delivery_days")
))

In [0]:
from pyspark.sql.functions import datediff,col
display(orders.select(
    datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    ).alias("delivery_days")
).describe())

In [0]:
display(orders.filter(
    col("order_status") == "delivered"
).filter(
    col("order_delivered_customer_date").isNull()
))